# NOK to EUR Currency Exchange Rates
This notebook fetches historical NOK (Norwegian Krone) to EUR (Euro) exchange rates from January 2018 to present.

**Data Source**: Fixer.io API (free tier) or ExchangeRate-API (free)
**Time Period**: January 2018 - Present
**Frequency**: Monthly data, then calculated quarterly averages
**Output**: CSV files with monthly and quarterly exchange rates

In [10]:
# Import required libraries
import pandas as pd
import numpy as np
import requests
import json
import time
from datetime import datetime, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("📚 Libraries imported successfully")
print(f"🕐 Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Configuration
data_path = Path('../data')
data_path.mkdir(exist_ok=True)

# Date range
START_DATE = "2018-01-01"
END_DATE = datetime.now().strftime("%Y-%m-%d")

print(f"📅 Date range: {START_DATE} to {END_DATE}")

📚 Libraries imported successfully
🕐 Current time: 2025-09-15 13:29:26
📅 Date range: 2018-01-01 to 2025-09-15


In [11]:
# Currency exchange rate fetching functions - Updated with working APIs

def get_exchange_rate_exchangerate_host(date_str, base="NOK", target="EUR"):
    """
    Fetch exchange rate from ExchangeRate.host (free, no API key required)
    Returns NOK to EUR rate for specific date
    """
    try:
        url = f"https://api.exchangerate.host/{date_str}"
        params = {
            'base': base,
            'symbols': target
        }
        
        response = requests.get(url, params=params, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            
            if data.get('success') and target in data.get('rates', {}):
                nok_to_eur = data['rates'][target]
                
                return {
                    'date': date_str,
                    'nok_to_eur': round(nok_to_eur, 6),
                    'source': 'exchangerate.host'
                }
            else:
                return None
        else:
            print(f"   ❌ API Error {response.status_code} for {date_str}")
            return None
            
    except Exception as e:
        print(f"   ⚠️ Error for {date_str}: {str(e)[:50]}")
        return None

def get_exchange_rate_frankfurter(date_str, base="NOK", target="EUR"):
    """
    Fetch exchange rate from Frankfurter API (free, no API key, ECB data)
    Returns NOK to EUR rate for specific date
    """
    try:
        url = f"https://api.frankfurter.app/{date_str}"
        params = {
            'from': base,
            'to': target
        }
        
        response = requests.get(url, params=params, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            
            if 'rates' in data and target in data['rates']:
                nok_to_eur = data['rates'][target]
                
                return {
                    'date': date_str,
                    'nok_to_eur': round(nok_to_eur, 6),
                    'source': 'frankfurter'
                }
            else:
                return None
        else:
            return None
            
    except Exception as e:
        print(f"   ⚠️ Frankfurter Error for {date_str}: {str(e)[:50]}")
        return None

def get_exchange_rate_vat_comply(date_str, base="NOK", target="EUR"):
    """
    Fetch exchange rate from VAT Comply API (free, no API key)
    Returns NOK to EUR rate for specific date
    """
    try:
        url = f"https://api.vatcomply.com/rates"
        params = {
            'date': date_str,
            'base': base
        }
        
        response = requests.get(url, params=params, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            
            if 'rates' in data and target in data['rates']:
                nok_to_eur = data['rates'][target]
                
                return {
                    'date': date_str,
                    'nok_to_eur': round(nok_to_eur, 6),
                    'source': 'vatcomply'
                }
            else:
                return None
        else:
            return None
            
    except Exception as e:
        print(f"   ⚠️ VAT Comply Error for {date_str}: {str(e)[:50]}")
        return None

def get_exchange_rate_fallback(date_str):
    """
    Fallback approach using multiple APIs with error handling
    """
    # Try APIs in order of preference
    apis = [
        ('Frankfurter (ECB)', get_exchange_rate_frankfurter),
        ('ExchangeRate.host', get_exchange_rate_exchangerate_host),
        ('VAT Comply', get_exchange_rate_vat_comply)
    ]
    
    for api_name, api_func in apis:
        try:
            result = api_func(date_str)
            if result:
                return result
        except:
            continue
    
    return None

def get_current_exchange_rate():
    """
    Get current NOK to EUR exchange rate using multiple sources
    """
    try:
        # Try Frankfurter for current rate
        url = "https://api.frankfurter.app/latest"
        params = {'from': 'NOK', 'to': 'EUR'}
        
        response = requests.get(url, params=params, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            if 'rates' in data and 'EUR' in data['rates']:
                return {
                    'date': data['date'],
                    'nok_to_eur': round(data['rates']['EUR'], 6),
                    'source': 'frankfurter-current'
                }
    except:
        pass
    
    # Fallback: use a typical recent rate if APIs fail
    return {
        'date': datetime.now().strftime('%Y-%m-%d'),
        'nok_to_eur': 0.085,  # Approximate recent NOK to EUR rate
        'source': 'fallback-estimate'
    }

print("✅ Updated currency exchange rate functions created")
print("🌐 Using multiple free APIs: Frankfurter (ECB), ExchangeRate.host, VAT Comply")
print("💱 Will fetch NOK to EUR conversion rates with fallback options")
print("🔄 Added error handling and multiple API sources")

✅ Updated currency exchange rate functions created
🌐 Using multiple free APIs: Frankfurter (ECB), ExchangeRate.host, VAT Comply
💱 Will fetch NOK to EUR conversion rates with fallback options
🔄 Added error handling and multiple API sources


In [12]:
# Test updated currency APIs
print("🧪 TESTING UPDATED CURRENCY APIS")
print("=" * 40)

# Test with a recent date
test_date = "2024-01-15"
print(f"Testing with date: {test_date}")

# Test Frankfurter API (ECB data, free)
print(f"\n1. Testing Frankfurter API (ECB data)...")
test_result1 = get_exchange_rate_frankfurter(test_date)
if test_result1:
    print(f"   ✅ Success: 1 NOK = {test_result1['nok_to_eur']:.6f} EUR")
else:
    print(f"   ❌ Failed")

# Test ExchangeRate.host
print(f"\n2. Testing ExchangeRate.host...")
test_result2 = get_exchange_rate_exchangerate_host(test_date)
if test_result2:
    print(f"   ✅ Success: 1 NOK = {test_result2['nok_to_eur']:.6f} EUR")
else:
    print(f"   ❌ Failed")

# Test VAT Comply
print(f"\n3. Testing VAT Comply API...")
test_result3 = get_exchange_rate_vat_comply(test_date)
if test_result3:
    print(f"   ✅ Success: 1 NOK = {test_result3['nok_to_eur']:.6f} EUR")
else:
    print(f"   ❌ Failed")

# Test fallback function
print(f"\n4. Testing fallback function...")
test_result4 = get_exchange_rate_fallback(test_date)
if test_result4:
    print(f"   ✅ Success: 1 NOK = {test_result4['nok_to_eur']:.6f} EUR (source: {test_result4['source']})")
else:
    print(f"   ❌ All APIs failed")

# Test current rate
print(f"\n5. Testing current exchange rate...")
current_result = get_current_exchange_rate()
if current_result:
    print(f"   ✅ Current rate: 1 NOK = {current_result['nok_to_eur']:.6f} EUR (date: {current_result['date']}, source: {current_result['source']})")

# Determine which API to use
working_apis = []
if test_result1: working_apis.append("Frankfurter")
if test_result2: working_apis.append("ExchangeRate.host") 
if test_result3: working_apis.append("VAT Comply")

if working_apis:
    print(f"\n✅ Working APIs found: {', '.join(working_apis)}")
    print(f"💡 Will use fallback function that tries multiple APIs")
else:
    print(f"\n⚠️ No APIs working - will use fallback estimates")

print(f"\n🎯 Ready to fetch monthly NOK to EUR rates from {START_DATE}")
print(f"🔄 Using robust fallback approach with multiple API sources")

🧪 TESTING UPDATED CURRENCY APIS
Testing with date: 2024-01-15

1. Testing Frankfurter API (ECB data)...
   ✅ Success: 1 NOK = 0.088390 EUR

2. Testing ExchangeRate.host...
   ✅ Success: 1 NOK = 0.088390 EUR

2. Testing ExchangeRate.host...
   ❌ Failed

3. Testing VAT Comply API...
   ❌ Failed

3. Testing VAT Comply API...
   ✅ Success: 1 NOK = 0.088386 EUR

4. Testing fallback function...
   ✅ Success: 1 NOK = 0.088386 EUR

4. Testing fallback function...
   ✅ Success: 1 NOK = 0.088390 EUR (source: frankfurter)

5. Testing current exchange rate...
   ✅ Success: 1 NOK = 0.088390 EUR (source: frankfurter)

5. Testing current exchange rate...
   ✅ Current rate: 1 NOK = 0.086320 EUR (date: 2025-09-12, source: frankfurter-current)

✅ Working APIs found: Frankfurter, VAT Comply
💡 Will use fallback function that tries multiple APIs

🎯 Ready to fetch monthly NOK to EUR rates from 2018-01-01
🔄 Using robust fallback approach with multiple API sources
   ✅ Current rate: 1 NOK = 0.086320 EUR (date

In [13]:
# Generate monthly dates and collect currency data with robust API handling

def generate_monthly_dates(start_date, end_date):
    """
    Generate first day of each month between start and end dates
    """
    start_dt = datetime.strptime(start_date, "%Y-%m-%d")
    end_dt = datetime.strptime(end_date, "%Y-%m-%d")
    
    dates = []
    current_date = start_dt.replace(day=1)  # First day of start month
    
    while current_date <= end_dt:
        dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to first day of next month
        if current_date.month == 12:
            current_date = current_date.replace(year=current_date.year + 1, month=1)
        else:
            current_date = current_date.replace(month=current_date.month + 1)
    
    return dates

def collect_monthly_currency_data():
    """
    Collect NOK to EUR exchange rates using robust fallback approach
    """
    print("💱 COLLECTING MONTHLY NOK TO EUR EXCHANGE RATES")
    print("=" * 60)
    
    # Generate monthly dates
    monthly_dates = generate_monthly_dates(START_DATE, END_DATE)
    print(f"📅 Will collect data for {len(monthly_dates)} months")
    print(f"   • From: {monthly_dates[0]}")
    print(f"   • To: {monthly_dates[-1]}")
    
    currency_data = []
    failed_dates = []
    api_sources = {}
    
    print(f"\n🔄 Starting currency data collection with fallback APIs...")
    print("-" * 60)
    
    for i, date_str in enumerate(monthly_dates, 1):
        print(f"[{i:3d}/{len(monthly_dates)}] Fetching rate for {date_str}...")
        
        # Use fallback function that tries multiple APIs
        rate_data = get_exchange_rate_fallback(date_str)
        
        if rate_data:
            currency_data.append(rate_data)
            source = rate_data['source']
            api_sources[source] = api_sources.get(source, 0) + 1
            print(f"   ✅ 1 NOK = {rate_data['nok_to_eur']:.6f} EUR (via {source})")
        else:
            # If all APIs fail, use interpolation based on successful rates
            if currency_data:
                # Use the last successful rate as estimate
                last_rate = currency_data[-1]['nok_to_eur']
                estimated_data = {
                    'date': date_str,
                    'nok_to_eur': last_rate,
                    'source': 'estimated'
                }
                currency_data.append(estimated_data)
                api_sources['estimated'] = api_sources.get('estimated', 0) + 1
                print(f"   ⚠️ Using estimated rate: {last_rate:.6f} EUR (APIs failed)")
            else:
                failed_dates.append(date_str)
                print(f"   ❌ Failed to get rate")
        
        # Rate limiting - be nice to free APIs
        time.sleep(1.0)  # Increased delay for stability
        
        # Progress update every 12 months
        if i % 12 == 0:
            print(f"   📊 Progress: {i}/{len(monthly_dates)} months, API sources: {dict(api_sources)}")
    
    print(f"\n✅ COLLECTION COMPLETE!")
    print(f"   • Successfully collected: {len(currency_data)} months")
    print(f"   • Failed requests: {len(failed_dates)}")
    print(f"   • Success rate: {(len(currency_data)/len(monthly_dates)*100):.1f}%")
    print(f"   • API source breakdown: {dict(api_sources)}")
    
    if failed_dates:
        print(f"   ⚠️ Failed dates: {failed_dates[:5]}{'...' if len(failed_dates) > 5 else ''}")
    
    return currency_data, failed_dates

# Show what we'll collect
monthly_dates = generate_monthly_dates(START_DATE, END_DATE)
print(f"📊 UPDATED CURRENCY DATA COLLECTION PLAN:")
print(f"   • Total months: {len(monthly_dates)}")
print(f"   • Date range: {monthly_dates[0]} to {monthly_dates[-1]}")
print(f"   • Sample dates: {monthly_dates[:3]} ... {monthly_dates[-3:]}")
print(f"   • Fallback APIs: Frankfurter, ExchangeRate.host, VAT Comply")
print(f"   • Estimated time: ~{len(monthly_dates)*1.0/60:.1f} minutes")
print(f"   • Error handling: Interpolation for failed requests")

📊 UPDATED CURRENCY DATA COLLECTION PLAN:
   • Total months: 93
   • Date range: 2018-01-01 to 2025-09-01
   • Sample dates: ['2018-01-01', '2018-02-01', '2018-03-01'] ... ['2025-07-01', '2025-08-01', '2025-09-01']
   • Fallback APIs: Frankfurter, ExchangeRate.host, VAT Comply
   • Estimated time: ~1.6 minutes
   • Error handling: Interpolation for failed requests


In [14]:
# Run the currency data collection
print("🚀 STARTING CURRENCY DATA COLLECTION")
print("=" * 50)

# Collect monthly currency data
currency_records, failed_dates = collect_monthly_currency_data()

if currency_records:
    # Convert to DataFrame
    currency_df = pd.DataFrame(currency_records)
    currency_df['date'] = pd.to_datetime(currency_df['date'])
    currency_df = currency_df.sort_values('date').reset_index(drop=True)
    
    print(f"\n📊 CURRENCY DATA ANALYSIS:")
    print(f"   • Total records: {len(currency_df)}")
    print(f"   • Date range: {currency_df['date'].min().strftime('%Y-%m-%d')} to {currency_df['date'].max().strftime('%Y-%m-%d')}")
    print(f"   • Exchange rate range: {currency_df['nok_to_eur'].min():.6f} to {currency_df['nok_to_eur'].max():.6f}")
    print(f"   • Average rate: {currency_df['nok_to_eur'].mean():.6f}")
    
    # Show sample data
    print(f"\n🔍 SAMPLE MONTHLY EXCHANGE RATES:")
    print(currency_df[['date', 'nok_to_eur']].head(10))
    
    # Save monthly data
    monthly_file = data_path / 'nok_eur_monthly.csv'
    currency_df.to_csv(monthly_file, index=False)
    print(f"\n💾 Monthly data saved to: {monthly_file}")
    
else:
    print(f"\n❌ No currency data collected!")
    print(f"Check API connectivity and try again.")

🚀 STARTING CURRENCY DATA COLLECTION
💱 COLLECTING MONTHLY NOK TO EUR EXCHANGE RATES
📅 Will collect data for 93 months
   • From: 2018-01-01
   • To: 2025-09-01

🔄 Starting currency data collection with fallback APIs...
------------------------------------------------------------
[  1/93] Fetching rate for 2018-01-01...
   ✅ 1 NOK = 0.101620 EUR (via frankfurter)
   ✅ 1 NOK = 0.101620 EUR (via frankfurter)
[  2/93] Fetching rate for 2018-02-01...
[  2/93] Fetching rate for 2018-02-01...
   ✅ 1 NOK = 0.104490 EUR (via frankfurter)
   ✅ 1 NOK = 0.104490 EUR (via frankfurter)
[  3/93] Fetching rate for 2018-03-01...
[  3/93] Fetching rate for 2018-03-01...
   ✅ 1 NOK = 0.103520 EUR (via frankfurter)
   ✅ 1 NOK = 0.103520 EUR (via frankfurter)
[  4/93] Fetching rate for 2018-04-01...
[  4/93] Fetching rate for 2018-04-01...
   ✅ 1 NOK = 0.103340 EUR (via frankfurter)
   ✅ 1 NOK = 0.103340 EUR (via frankfurter)
[  5/93] Fetching rate for 2018-05-01...
[  5/93] Fetching rate for 2018-05-01...


In [15]:
# Convert monthly data to quarterly averages
def create_quarterly_data(monthly_df):
    """
    Convert monthly currency data to quarterly averages
    """
    print("📊 CONVERTING TO QUARTERLY AVERAGES")
    print("=" * 40)
    
    # Add year and quarter columns
    quarterly_df = monthly_df.copy()
    quarterly_df['year'] = quarterly_df['date'].dt.year
    quarterly_df['quarter'] = quarterly_df['date'].dt.quarter
    quarterly_df['year_quarter'] = quarterly_df['year'].astype(str) + '-Q' + quarterly_df['quarter'].astype(str)
    
    # Calculate quarterly averages
    quarterly_summary = quarterly_df.groupby(['year', 'quarter']).agg({
        'nok_to_eur': ['mean', 'min', 'max', 'count'],
        'date': ['min', 'max']
    }).round(6)
    
    # Flatten column names
    quarterly_summary.columns = [
        'avg_nok_to_eur', 'min_nok_to_eur', 'max_nok_to_eur', 'months_count',
        'quarter_start', 'quarter_end'
    ]
    
    # Reset index and create readable quarter labels
    quarterly_summary = quarterly_summary.reset_index()
    quarterly_summary['year_quarter'] = quarterly_summary['year'].astype(str) + '-Q' + quarterly_summary['quarter'].astype(str)
    
    # Reorder columns
    quarterly_summary = quarterly_summary[[
        'year', 'quarter', 'year_quarter', 'quarter_start', 'quarter_end',
        'avg_nok_to_eur', 'min_nok_to_eur', 'max_nok_to_eur', 'months_count'
    ]]
    
    print(f"✅ Created quarterly data:")
    print(f"   • Total quarters: {len(quarterly_summary)}")
    print(f"   • Years covered: {quarterly_summary['year'].min()} to {quarterly_summary['year'].max()}")
    print(f"   • Average exchange rate: {quarterly_summary['avg_nok_to_eur'].mean():.6f}")
    
    return quarterly_summary

# Create quarterly data if we have monthly data
if 'currency_df' in locals() and len(currency_df) > 0:
    quarterly_df = create_quarterly_data(currency_df)
    
    print(f"\n🔍 SAMPLE QUARTERLY DATA:")
    print(quarterly_df[['year_quarter', 'quarter_start', 'avg_nok_to_eur', 'months_count']].head(10))
    
    print(f"\n📈 QUARTERLY EXCHANGE RATE TRENDS:")
    print(f"   • Highest rate: {quarterly_df['avg_nok_to_eur'].max():.6f} (Q{quarterly_df.loc[quarterly_df['avg_nok_to_eur'].idxmax(), 'year_quarter']})")
    print(f"   • Lowest rate: {quarterly_df['avg_nok_to_eur'].min():.6f} (Q{quarterly_df.loc[quarterly_df['avg_nok_to_eur'].idxmin(), 'year_quarter']})")
    print(f"   • Overall average: {quarterly_df['avg_nok_to_eur'].mean():.6f}")
    
    # Show recent quarters
    print(f"\n📅 RECENT QUARTERLY RATES:")
    print(quarterly_df[['year_quarter', 'avg_nok_to_eur']].tail(8))
    
    # Save quarterly data
    quarterly_file = data_path / 'nok_eur_quarterly.csv'
    quarterly_df.to_csv(quarterly_file, index=False)
    print(f"\n💾 Quarterly data saved to: {quarterly_file}")
    
else:
    print(f"\n⚠️ No monthly data available to convert to quarterly")

📊 CONVERTING TO QUARTERLY AVERAGES
✅ Created quarterly data:
   • Total quarters: 31
   • Years covered: 2018 to 2025
   • Average exchange rate: 0.094889

🔍 SAMPLE QUARTERLY DATA:
  year_quarter quarter_start  avg_nok_to_eur  months_count
0      2018-Q1    2018-01-01        0.103210             3
1      2018-Q2    2018-04-01        0.103917             3
2      2018-Q3    2018-07-01        0.104307             3
3      2018-Q4    2018-10-01        0.104543             3
4      2019-Q1    2019-01-01        0.102253             3
5      2019-Q2    2019-04-01        0.103110             3
6      2019-Q3    2019-07-01        0.101597             3
7      2019-Q4    2019-10-01        0.099300             3
8      2020-Q1    2020-01-01        0.098593             3
9      2020-Q2    2020-04-01        0.090280             3

📈 QUARTERLY EXCHANGE RATE TRENDS:
   • Highest rate: 0.104543 (Q2018-Q4)
   • Lowest rate: 0.084773 (Q2025-Q3)
   • Overall average: 0.094889

📅 RECENT QUARTERLY RATES:
